# Grover's search on three qubits

| | |
|---|---|
| **Level** | Introductory to intermediate |
| **Time** | About 45 minutes |
| **Prerequisites** | Hadamard, X, and multi-controlled gates. Amplitudes and interference help. |
| **Default device** | IQM Garnet |
| **Also runs on** | Rigetti Cepheus-1-108Q, IonQ Forte Enterprise (about 830 credits per job) |
| **Qubits** | 3 |
| **Two-qubit gates** | about 12 for one iteration before routing, about 19 after routing on a square lattice |
| **Hardware jobs** | 1 (or 2, see Settings) |
| **Approximate cost** | Garnet at 500 shots: about 103 credits per iteration run. Rigetti: about 10 credits per job. |
| **Suggested hand-in** | Your bar chart, and answers to Questions 1 and 2 |

The device, the number of shots, and whether to use hardware are set in the **Settings** cell below. Nothing is sent to hardware until you set `RUN_ON_HARDWARE = True`.

*Part of the QUEST notebooks from qBraid. You may copy, edit and adapt this notebook for your course.*

Grover's algorithm finds a marked item among $N$ possibilities using about $\sqrt{N}$ queries, where a classical search needs about $N/2$ on average.

Each **Grover iteration** has two steps. The *oracle* flips the sign of the marked state's amplitude. The *diffuser* reflects every amplitude about the average, which increases the marked amplitude and shrinks the others. After $k$ iterations on $n$ qubits, the probability of finding the marked item is

$$P(k) = \sin^2\big((2k+1)\theta\big), \qquad \sin\theta = 1/\sqrt{2^n}.$$

With three qubits there are 8 items. One iteration gives 0.781, two give 0.945, and random guessing gives 0.125. Three qubits is a deliberate choice: one iteration is short enough to run well on today's hardware, and two iterations show what happens when a circuit gets too long.

In [ ]:
# Settings. Change these, then run the notebook from the top.
DEVICE_ID = "aws:iqm:qpu:garnet"   # device list and prices: see the README
SHOTS = 500                  # measurements per hardware job
RUN_ON_HARDWARE = False     # set to True when you are ready to spend credits
ITERATIONS_ON_HARDWARE = [1]   # add 2 to see a longer circuit on hardware (one job each)
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']   # gates every QUEST device accepts
QUEST_JOB_TAGS = {"quest": "algo-grover3"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qbraid.runtime import QbraidProvider

In [ ]:
# Helper functions. You do not need to read these to follow the notebook.

def estimate_cost(device, n_jobs, shots):
    """Estimated qBraid credits for n_jobs hardware jobs (100 credits = $1)."""
    pricing = getattr(device.profile, "pricing", None)
    if pricing is None:
        return "no published price for this device"
    if float(pricing.perMinute) > 0:
        return ("billed by execution time, so it cannot be quoted in advance "
                f"({float(pricing.perMinute):,.0f} credits per minute of device time; "
                "small jobs cost about 10 credits each in our tests)")
    per_job = float(pricing.perTask) + float(pricing.perShot) * shots
    return f"about {n_jobs * per_job:,.0f} credits ({n_jobs} job(s) at {per_job:,.1f} each)"


def prob_one(counts, qubit, n_qubits):
    """Fraction of shots in which `qubit` was measured as 1. Qubit 0 is the rightmost bit."""
    total = sum(counts.values())
    ones = 0
    for key, n in counts.items():
        bits = key.replace(" ", "").zfill(n_qubits)
        if bits[-(qubit + 1)] == "1":
            ones += n
    return ones / total

## 1. Build the circuit

The marked item is `MARKED`. Qiskit writes qubit 0 as the rightmost bit.

In [ ]:
MARKED = "101"
N_QUBITS = 3

def ccz(qc):
    """Flip the sign of |111>: a doubly controlled Z, built from H and a Toffoli."""
    qc.h(2)
    qc.ccx(0, 1, 2)
    qc.h(2)

def oracle(qc):
    """Flip the sign of |MARKED>."""
    zeros = [q for q, bit in enumerate(reversed(MARKED)) if bit == "0"]
    for q in zeros:
        qc.x(q)
    ccz(qc)
    for q in zeros:
        qc.x(q)

def diffuser(qc):
    """Reflect all amplitudes about their average."""
    qc.h(range(N_QUBITS))
    qc.x(range(N_QUBITS))
    ccz(qc)
    qc.x(range(N_QUBITS))
    qc.h(range(N_QUBITS))

def grover(iterations):
    qc = QuantumCircuit(N_QUBITS)
    qc.h(range(N_QUBITS))
    for _ in range(iterations):
        oracle(qc)
        diffuser(qc)
    qc.measure_all()
    return qc

grover(1).draw(output="text", fold=120)

## 2. Ideal simulation

In [ ]:
theta = np.arcsin(1 / np.sqrt(2 ** N_QUBITS))
simulator = AerSimulator()

def p_marked(counts):
    total = sum(counts.values())
    return sum(n for key, n in counts.items() if key.replace(" ", "").zfill(N_QUBITS) == MARKED) / total

iterations = [0, 1, 2, 3, 4]
ideal_p = [p_marked(simulator.run(grover(k), shots=SHOTS).result().get_counts()) for k in iterations]
for k, p in zip(iterations, ideal_p):
    print(f"k = {k}: simulated {p:.3f}   predicted {np.sin((2 * k + 1) * theta) ** 2:.3f}")

The probability rises to a maximum at two iterations, then falls. Doing more iterations than needed rotates past the answer.

## 3. Run on hardware

In [ ]:
hw_circuits = {k: transpile(grover(k), basis_gates=HW_BASIS, optimization_level=1)
               for k in ITERATIONS_ON_HARDWARE}
for k, qc in hw_circuits.items():
    ops = qc.count_ops()
    print(f"k = {k}: two-qubit gates before routing = {ops.get('cx', 0) + ops.get('cz', 0)}")

In [ ]:
N_JOBS = len(hw_circuits)

# Connect to the device and estimate the cost. This step is free.
try:
    provider = QbraidProvider()
    device = provider.get_device(DEVICE_ID)
    status = device.status().name
    print(f"Device:         {DEVICE_ID}")
    print(f"Status:         {status}")
    print(f"Estimated cost: {estimate_cost(device, n_jobs=N_JOBS, shots=SHOTS)}")
    if status != "ONLINE":
        print("This device is not online right now. Some devices run in scheduled windows.")
        print("Try again later, or choose another device in Settings.")
except Exception as err:
    device = None
    print(f"Could not reach qBraid ({err}). The simulation sections still work.")

In [ ]:
if RUN_ON_HARDWARE and device is not None:
    jobs = {k: device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS) for k, qc in hw_circuits.items()}
    print("Submitted. Waiting for results; queues can take anywhere from seconds to hours.")
    hw_counts = {k: job.result().data.get_counts() for k, job in jobs.items()}
    for k in hw_counts:
        print(f"k = {k}: P(marked) = {p_marked(hw_counts[k]):.3f}")
else:
    hw_counts = None
    print("Hardware step skipped. Set RUN_ON_HARDWARE = True in Settings to run it.")

## 4. Compare

In [ ]:
x = np.array(iterations)
plt.bar(x - 0.2, ideal_p, width=0.4, color="gray", label="ideal simulation")
if hw_counts:
    ks = sorted(hw_counts)
    plt.bar(np.array(ks) + 0.2, [p_marked(hw_counts[k]) for k in ks], width=0.4,
            color="tab:orange", label=f"hardware ({DEVICE_ID})")
plt.axhline(1 / 8, color="black", linestyle=":", label="random guessing (1/8)")
plt.xticks(x)
plt.xlabel("Grover iterations k")
plt.ylabel("probability of finding the marked item")
plt.legend()
plt.show()

## Questions to try

1. Is the hardware result well above random guessing (0.125)? By how much does it fall short of the ideal 0.781?
2. Two iterations should give 0.945. Add 2 to `ITERATIONS_ON_HARDWARE` and run again. Does two iterations beat one on hardware? Use the two-qubit gate counts to explain what you see.
3. Change `MARKED` to `"000"` and then to `"111"`. Does the hardware find one of them more easily? [the readout-error notebook](../foundations/intro_01_measuring_readout_error.ipynb) and [the noise notebook](../noise_and_hardware/intro_01_noise_on_real_hardware.ipynb) may suggest a reason.
4. Four-qubit Grover (16 items) needs about 35 two-qubit gates after routing for a single iteration, several times more than this circuit. What would you expect on hardware, and what does that suggest about the size of problem today's devices can run?